# FINA 4075/5075 — Project 2 · The Strategy Lab — pre-registration notebook (Fall 2026)

**Repo:** FINTECH-P2-Example · **Track 1:** intermediate momentum, months t−12 … t−7 (top 20 of 200, equal weights, 10 bp per side) · **Track 2:** frozen prompt on Claude Fable 5.1 Max · **Pre-registration commit:** Sun 9/20/2026 (hash quoted in `monitoring_log.md` and the final report)

**Before running anything:** File → **Save a copy in Drive**, then Runtime → Change runtime type → **Runtime version 2026.07** (Rule 0). Cells 1–4 are the Project 0 template. Run top to bottom; finish with **Runtime → Restart session and run all** (Rule 6) before you save the notebook to GitHub.

**Code Rule:** after the 9/20 commit the strategy code (`strategy.py`, written by Cell 6) is read-only. At each checkpoint you point Cell 2 at the new dated extension file and rerun — nothing else changes.

In [ ]:
# Cell 1 — mount Drive (Rule 1)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Cell 2 — course folder and the packet files (Rule 1 / Rule 4: analysis reads the dated snapshot)
import os
P2 = "/content/drive/MyDrive/FINA4075/P2"
OUT = P2 + "/outputs"
os.makedirs(OUT, exist_ok=True)

PANEL_FILE = P2 + "/panel_returns_2026-08-31.csv"      # at checkpoints: the dated extension file (e.g. ..._2026-09-30.csv)
TICKER_FILE = P2 + "/P2_ticker_list_2026-08-31.csv"
PANEL_SHA256 = "c9d3340eba3a3301a63bb788c87f1d6e9e55de58a3fec16b12a34b313cac076b"   # from the Data Dictionary
print("Folder ready:", P2)

In [ ]:
# Cell 3 — pinned setup (Rule 2). Official pinned line from D2L; the P2 analysis itself needs only pandas, numpy and
# matplotlib, which the pinned runtime already provides — never reinstall those.
%pip install -q yfinance==0.2.66

In [ ]:
# Cell 4 — environment stamp (Rule 0)
import sys, pandas as pd, numpy as np, matplotlib, yfinance as yf
print(sys.version)
print("pandas", pd.__version__, "| numpy", np.__version__,
      "| matplotlib", matplotlib.__version__, "| yfinance", yf.__version__)

## Step 1 — load the packet and verify the hash
The SHA-256 of the panel file must equal the value published in the Data Dictionary. If it does not, the file was edited or re-saved: download it again from D2L.

In [ ]:
# Cell 5 — hash check and load
import hashlib
sha = hashlib.sha256(open(PANEL_FILE, "rb").read()).hexdigest()
print("SHA-256:", sha)
assert sha == PANEL_SHA256, "panel file does not match the Data Dictionary hash"

panel = pd.read_csv(PANEL_FILE, index_col=0, parse_dates=True)
info = pd.read_csv(TICKER_FILE).set_index("Ticker")
print("panel shape:", panel.shape, "| first month:", panel.index[0].date(), "| last month:", panel.index[-1].date())
print("missing values:", int(panel.isna().sum().sum()), "| tickers match the company list:", list(panel.columns) == list(info.index))

## Steps 4 and 7 — the locked rule
The rule lives in one plain-text file, `strategy.py`, so the grader can diff it against the 9/20 commit. Cell 6 writes the file into the P2 folder; Cell 7 imports it. `strategy_rules.md` governs if the two ever disagree.

In [ ]:
%%writefile /content/drive/MyDrive/FINA4075/P2/strategy.py
"""
strategy.py — LOCKED STRATEGY CODE (Track 1)   ·   FINTECH-P2-Example   ·   FINA 4075/5075, Fall 2026

Rule (see strategy_rules.md, which governs if the two ever disagree):
  Intermediate momentum, months t-12 through t-7.  For the portfolio held in month t, rank the 200
  stocks by their compounded total return over the SIX monthly returns t-12, t-11, ..., t-7
  (the most recent six months, t-6 .. t-1, are skipped entirely).  Hold the top 20, equal weights,
  rebalanced at every month-end from data through the prior month-end only.  10 bp per side.

CODE RULE: this file is read-only after the 9/20/2026 pre-registration commit.  At each checkpoint
the panel is extended by one month and this code is rerun unchanged.
"""
import numpy as np
import pandas as pd

PER_SIDE = 0.0010        # transaction cost, 10 basis points per side
N_HOLD = 20              # holding count (the top decile of the 200-stock panel)
WINDOW_START = 12        # window starts WINDOW_START months before month t  (t-12)
WINDOW_END = 7           # window ends   WINDOW_END   months before month t  (t-7), inclusive
FIRST_TRADEABLE = WINDOW_START   # the first month t whose window exists is row 12 (January 2020)


def load_panel(panel_path, ticker_path):
    """Read the course packet.  Returns (rets, dates, tickers, info).

    rets    : numpy array, T x N, simple monthly total returns (row 0 = January 2019)
    dates   : the month-end dates of the rows
    tickers : list of the N ticker symbols, in the column order of the panel
    info    : the company list (ticker, company, sector, ...) indexed by ticker
    """
    panel = pd.read_csv(panel_path, index_col=0, parse_dates=True)
    info = pd.read_csv(ticker_path).set_index("Ticker")
    tickers = list(panel.columns)
    assert list(info.index) == tickers, "ticker list and panel columns differ"
    assert not panel.isna().any().any(), "panel has missing values"
    return panel.to_numpy(), panel.index, tickers, info


def signal(t, rets):
    """Compounded return over rows t-12 .. t-7 inclusive (six monthly returns).

    Python slicing excludes the stop index, so rows t-12 .. t-7 are rets[t-12 : t-6].
    """
    window = rets[t - WINDOW_START : t - WINDOW_END + 1]
    assert window.shape[0] == WINDOW_START - WINDOW_END + 1 == 6
    return np.prod(1 + window, axis=0) - 1


def select_holdings(t, rets, tickers):
    """The 20 stocks held during month t: highest signal first; ties broken by alphabetical ticker.

    A stock with any missing return in its t-12..t-7 window is ineligible for month t (rules: MISSING
    DATA). The released panel has no gaps, so every stock is eligible in-sample; this guard governs
    the live window, where an extension file could carry a delisted name's partial history.
    """
    window = rets[t - WINDOW_START : t - WINDOW_END + 1]
    eligible = ~np.isnan(window).any(axis=0)
    sig = signal(t, rets)
    cand = [i for i in range(len(tickers)) if eligible[i]]
    order = sorted(cand, key=lambda i: (-sig[i], tickers[i]))
    return set(order[:N_HOLD])


def run_track(select_fn, rets, t0, t1, per_side=PER_SIDE):
    """Paper-trade a selection rule from month t0 to month t1 (row indices, inclusive).

    Course conventions: equal weights; one-way turnover = names replaced / 20 (the first month
    counts as a full purchase); monthly cost = 2 x per_side x turnover; growth path starts at 1.0.
    Returns (values, turnovers, holdings) with len(values) == months + 1.
    """
    prev, vals, turns, holdings = set(), [1.0], [], []
    for t in range(t0, t1 + 1):
        hold = select_fn(t, rets)
        turn = 1.0 if not prev else len(hold - prev) / N_HOLD
        gross = rets[t, sorted(hold)].mean()
        vals.append(vals[-1] * (1 + gross - 2 * per_side * turn))
        prev = hold
        turns.append(turn)
        holdings.append(hold)
    return np.array(vals), np.array(turns), holdings


def bench(rets, t0, t1):
    """Equal-weighted average of all 200 stocks, rebalanced monthly, no costs."""
    vals = [1.0]
    for t in range(t0, t1 + 1):
        vals.append(vals[-1] * (1 + rets[t].mean()))
    return np.array(vals)


def stats(vals, turns=None):
    """CAGR, annualized volatility, Sharpe (rf = 0), maximum drawdown, annual one-way turnover."""
    months = len(vals) - 1
    monthly = vals[1:] / vals[:-1] - 1
    cagr = vals[-1] ** (12 / months) - 1
    vol = monthly.std(ddof=1) * np.sqrt(12)
    maxdd = (vals / np.maximum.accumulate(vals) - 1).min()
    out = {"CAGR": cagr, "Annualized volatility": vol, "Sharpe (rf = 0)": cagr / vol,
           "Maximum drawdown": maxdd, "Growth of $10,000": 10000 * vals[-1]}
    if turns is not None:
        out["Annual one-way turnover"] = turns.mean() * 12
        out["Cost drag per year"] = 2 * PER_SIDE * turns.mean() * 12
    return out


In [ ]:
# Cell 7 — import the locked code from Drive and bind the rule to this panel
import importlib, sys
if P2 not in sys.path:
    sys.path.insert(0, P2)
import strategy as S
importlib.reload(S)

rets, dates, tickers, info = S.load_panel(PANEL_FILE, TICKER_FILE)
def rule(t, rets):
    return S.select_holdings(t, rets, tickers)
print("rows (months):", len(dates), "| first tradeable row:", S.FIRST_TRADEABLE, "=", dates[S.FIRST_TRADEABLE].date())

## Step 7 — in-sample backtest, January 2020 – August 2026 (80 months)
Pre-live sample only. Net of 10 bp per side; the benchmark is the equal-weighted panel with no costs; rf = 0 for the Sharpe ratio.

In [ ]:
# Cell 8 — run both tracks' in-sample paths and the stats table
t0, t1 = S.FIRST_TRADEABLE, len(dates) - 1                 # row 12 (Jan 2020) .. row 91 (Aug 2026)
vals, turns, holdings = S.run_track(rule, rets, t0, t1)
bvals = S.bench(rets, t0, t1)

strat_stats, bench_stats = S.stats(vals, turns), S.stats(bvals)
rows = ["CAGR", "Annualized volatility", "Sharpe (rf = 0)", "Maximum drawdown", "Annual one-way turnover", "Cost drag per year", "Growth of $10,000"]
table = pd.DataFrame({"Intermediate momentum (net)": [strat_stats.get(r) for r in rows],
                      "EW benchmark (no costs)": [bench_stats.get(r) for r in rows]}, index=rows)
def fmt(v, r):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "—"
    if r == "Growth of $10,000": return f"${v:,.0f}"
    if r == "Sharpe (rf = 0)": return f"{v:.2f}"
    return f"{v*100:.1f}%"
shown = pd.DataFrame({c: [fmt(table.loc[r, c], r) for r in rows] for c in table.columns}, index=rows)
print(f"In-sample window: {dates[t0].date()} to {dates[t1].date()} ({t1 - t0 + 1} months)")
print(shown.to_string())
table.to_csv(OUT + "/insample_stats_2026-08-31.csv")

In [ ]:
# Cell 9 — growth-of-$10,000 exhibit (saved to Drive, Rule 7)
import matplotlib.pyplot as plt
x = dates[t0 - 1 : t1 + 1]                                 # month-ends; the path starts one month before the first holding month
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(x, 10000 * vals, label="Track 1: intermediate momentum (t−12…t−7), top 20 EW, net of 10 bp/side", lw=1.8)
ax.plot(x, 10000 * bvals, label="Equal-weighted panel (200 stocks), no costs", lw=1.8)
ax.axhline(10000, ls="--", lw=0.8, color="gray")
ax.set_title("Growth of $10,000 — in-sample, January 2020 to August 2026 (80 months)")
ax.set_ylabel("Portfolio value ($)")
ax.legend(loc="upper left", fontsize=9)
ax.text(0.01, -0.16, "Source: panel_returns_2026-08-31.csv (Yahoo Finance, dividends reinvested). Panel has survivorship and selection bias; rf = 0 for Sharpe.",
        transform=ax.transAxes, fontsize=7.5, color="gray")
plt.tight_layout()
fig.savefig(OUT + "/insample_growth_2026-08-31.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 7 — hand verification of one month (August 2026)
Pick three of the twenty holdings, look their August 2026 returns up in the CSV, and confirm the arithmetic below: gross = the simple average of the twenty returns; cost = 2 × 0.0010 × turnover; net = gross − cost.

In [ ]:
# Cell 10 — one month by hand: the last in-sample month, August 2026 (row 91)
t = t1
hold_now, hold_prev = holdings[t - t0], holdings[t - t0 - 1]
names = sorted(tickers[i] for i in hold_now)
r_month = pd.Series({tickers[i]: rets[t, i] for i in hold_now}).sort_index()
gross = r_month.mean()
turn = len(hold_now - hold_prev) / S.N_HOLD
cost = 2 * S.PER_SIDE * turn
print(f"Month scored: {dates[t].date()} | signal window: {dates[t-12].date()} .. {dates[t-7].date()}")
print("Holdings (20):", ", ".join(names))
print((100 * r_month).round(2).to_string())
print(f"\ngross = mean of the 20 returns = {gross:.6f} ({gross*100:.3f}%)")
print(f"turnover = names replaced / 20 = {len(hold_now - hold_prev)} / 20 = {turn:.2f}  ->  cost = 2 x 0.0010 x {turn:.2f} = {cost:.6f}")
print(f"net = {gross - cost:.6f} ({(gross - cost)*100:.3f}%)  ->  on $10,000: ${10000 * (gross - cost):,.2f}")
print("Three holdings to check against the CSV by hand:", ", ".join(f"{n} {rets[t, tickers.index(n)]*100:+.3f}%" for n in names[:3]))
print("Engine's own value for this month:", f"{vals[t - t0 + 1] / vals[t - t0] - 1:.6f}", "(must equal net above)")

## Step 8 — the briefing-table generator (the only thing Track 2 ever sees)
Format per the Briefing-Table Format Specification: 200 rows sorted by ticker; `Ticker | Company | Sector | 1M | 3M | 12M`, trailing total returns in percent (one decimal) as of the stated month-end. The text block is saved to Drive and pasted, unedited, under the frozen prompt.

In [ ]:
# Cell 11 — briefing table as of a month-end (default: the last row of the panel)
def briefing_table(panel, info, as_of=None):
    r = panel if as_of is None else panel.loc[:as_of]
    as_of = r.index[-1].date()
    r1 = r.iloc[-1]
    r3 = (1 + r.iloc[-3:]).prod() - 1
    r12 = (1 + r.iloc[-12:]).prod() - 1
    tbl = pd.DataFrame({"Ticker": r.columns,
                        "Company": info.loc[r.columns, "Company"].values,
                        "Sector": info.loc[r.columns, "Sector"].values,
                        "1M": (100 * r1.values).round(1) + 0.0, "3M": (100 * r3.values).round(1) + 0.0, "12M": (100 * r12.values).round(1) + 0.0})   # + 0.0 turns -0.0 into 0.0
    return tbl.sort_values("Ticker").reset_index(drop=True), as_of

def briefing_text(tbl, as_of):
    lines = [f"BRIEFING TABLE as of {as_of} (month-end). Trailing total returns in percent, dividends reinvested. {len(tbl)} stocks.",
             "Ticker | Company | Sector | 1M | 3M | 12M"]
    for row in tbl.itertuples(index=False):
        lines.append(f"{row.Ticker} | {row.Company} | {row.Sector} | {row[3]:+.1f} | {row[4]:+.1f} | {row[5]:+.1f}")
    return "\n".join(lines)

tbl, as_of = briefing_table(panel, info)                 # August 2026 month-end for the 9/20 commit
text = briefing_text(tbl, as_of)
open(OUT + f"/briefing_table_{as_of}.txt", "w").write(text)
tbl.to_csv(OUT + f"/briefing_table_{as_of}.csv", index=False)
print(text)

## Step 9 — entry 0: Track 1's holdings for September 2026
September 2026 is row 92, one past the end of the released panel, so its signal window is September 2025 – February 2026 (rows 80–85). The alphabetical list below goes into `monitoring_log.md` entry 0 next to the frozen prompt's picks.

In [ ]:
# Cell 12 — the locked rule's September 2026 portfolio, from data through August 2026
t_next = len(dates)                                        # row 92 = September 2026
hold_next = rule(t_next, rets)
sig_next = S.signal(t_next, rets)
sep = sorted(tickers[i] for i in hold_next)
print(f"Signal window: {dates[t_next-12].date()} .. {dates[t_next-7].date()} (six months)")
print("TRACK 1 September 2026 holdings (alphabetical):")
print(", ".join(sep))
ranked = pd.Series(sig_next, index=tickers).sort_values(ascending=False)
print("\nRanks 18-23 around the boundary (signal = compounded 6-month return):")
print((100 * ranked.iloc[17:23]).round(2).to_string())
open(OUT + "/track1_holdings_2026-09.txt", "w").write(", ".join(sep) + "\n")
print("\nSaved:", sorted(os.listdir(OUT)))

## Before the commit (Sun 9/20)
1. Runtime → **Restart session and run all** — no errors, then save.
2. Paste Track 1's list (Cell 12) and the frozen prompt's verbatim output into `monitoring_log.md` entry 0.
3. Commit to the private repo FINTECH-P2-Example: `strategy_rules.md`, `strategy.py` (Cell 6) with this notebook (File → **Save a copy in GitHub**), `llm_analyst_prompt.md`, `redteam_log.md`, `monitoring_log.md`, and `panel_returns_2026-08-31.csv` (its hash must match the Data Dictionary).
4. Copy the seven-character commit hash and quote it in every later deliverable.